# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a Croissant FAIR^2 dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/api/python/) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant JSON-LD schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Extract high-level metadata attributes
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")
print(f"Published: {metadata.datePublished}\nAuthors: {[str(a) for a in (getattr(metadata, 'author', []))]}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

All references to data entities use the `@id` field following best FAIR practices.

In [ ]:
# List all record sets with their @id and available fields
record_sets = dataset.metadata.record_sets
print("Available Record Sets:")
all_field_ids = {}
for rs in record_sets:
    print(f"- Record Set @id: {rs['@id']} | name: {rs.get('name', '<no name>')}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    field_ids = []
    print("    Fields:")
    for f in fields:
        f_id = f['@id']
        field_ids.append(f_id)
        print(f"        - Field @id: {f_id} | name: {f.get('name', '<no name>')}")
    all_field_ids[rs['@id']] = field_ids

// You can use these @ids below to refer to record set(s) and field(s)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Choose a record set to extract (replace with desired @id from above, example shown)
# Here we auto-select the first record set for demonstration purposes
selected_record_sets = [rs['@id'] for rs in record_sets]
if not selected_record_sets:
    raise ValueError("No record sets found in the metadata.")
dataframes = {}
for record_set_id in selected_record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded record set: {record_set_id}, DataFrame shape: {df.shape}")

# Display field (column) names (replace selected_record_sets[0] as needed)
print(f"Columns in record set {selected_record_sets[0]}:")
print(dataframes[selected_record_sets[0]].columns.tolist())
# Show the first 5 rows as preview
dataframes[selected_record_sets[0]].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

You can copy actual `@id`s of fields from earlier cells (section 2).

In [ ]:
# Select the main record set and peek at numeric fields
record_set_id = selected_record_sets[0]
df = dataframes[record_set_id]

# List numeric fields by a simple heuristic: detect numeric dtypes or columns with all numbers
numeric_cols = [col for col in df.columns if (pd.api.types.is_numeric_dtype(df[col]) or df[col].apply(lambda x: pd.api.types.is_number(x) if pd.notnull(x) else False).all())]
if not numeric_cols:
    print("No numeric columns detected.")
else:
    print("Numeric fields detected:", numeric_cols)

# For demo, pick the first numeric field, or assign manually by @id:
if numeric_cols:
    numeric_field_id = numeric_cols[0]   # replace with actual @id as needed
    threshold = df[numeric_field_id].mean()  # Example threshold (mean value)
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}: {len(filtered_df)} rows")
    print(filtered_df.head())
    
    # Normalize the field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Top 5 normalized values for {numeric_field_id}:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a categorical field, if present
    # Heuristically: next non-numeric column
    cat_cols = [col for col in df.columns if col != numeric_field_id and not pd.api.types.is_numeric_dtype(df[col])]
    if cat_cols:
        group_field_id = cat_cols[0]
        print(f"Grouping by: {group_field_id}")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(grouped_df.head())
    else:
        print("No suitable categorical field found for grouping.")
else:
    print("Unable to continue EDA: No numeric fields found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. For example, plot the distribution of a numeric field or the group means (replace field `@id` as desired).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the first numeric field in the main record set
if 'numeric_field_id' in locals():
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
# If grouping was performed, plot group means
if 'grouped_df' in locals():
    plt.figure(figsize=(10,5))
    sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id)
    plt.title(f'Average {numeric_field_id} by {group_field_id}')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR^2 dataset using `mlcroissant` via its Croissant schema, listing all available record sets and their fields by `@id`, extracting records into DataFrames, running basic EDA (including filtering and normalization by numeric field `@id`), and visualizing distributions.

The use of Croissant `@id` ensures reproducibility and semantic clarity when referencing dataset elements across analysis workflows.